# MADS Synthetic Dataset Generator

Generates a synthetic cybersecurity event dataset for evaluating the **Malicious Alert Detection System (MADS)**.

## Features
- **Configurable event count** (`n_events` in CONFIG) across 5 machines with realistic Dirichlet-distributed counts
- **65 / 35 malicious / benign** split with per-machine probability tuning
- **Multistage attack chains** (recon → delivery → execution → persistence → exfiltration)
- **Modular LLM backends** — choose one:

| Backend | `llm_backend` value | Package needed |
|---------|---------------------|----------------|
| No LLM (default) | `"synthetic"` | — |
| OpenAI ChatGPT | `"openai"` | `openai` |
| Cohere Command-A | `"cohere"` | `cohere` |

Set `llm_backend` and the corresponding API key in the **Configuration** cell before running.

In [2]:
# Install required packages (run once; comment out afterwards)
%pip install --quiet numpy pandas openai cohere tqdm

Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import random
import warnings
from abc import ABC, abstractmethod
from datetime import datetime, timedelta
from typing import Optional

from tqdm.auto import tqdm

import numpy as np
import pandas as pd


warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1200)

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CONFIGURATION  ← edit this cell before running
# ─────────────────────────────────────────────────────────────────────────────
CONFIG = {
    # ── Dataset size ──────────────────────────────────────────────────────────
    "n_events": 6000,

    # ── Machines in the network ───────────────────────────────────────────────
    "machines": ["machine1", "machine2", "machine3", "machine4", "machine5"],

    # Per-machine probability that a generated event is malicious.
    # machine1 = mostly malicious, machine5 = mostly benign.
    "machine_malicious_probs": {
        "machine1": 0.80,
        "machine2": 0.75,
        "machine3": 0.65,
        "machine4": 0.59,
        "machine5": 0.20,
    },

    # Dirichlet concentration params for machine event-count distribution.
    # Higher value → more events assigned to that machine index.
    "machine_dirichlet_alphas": [7, 5, 5, 4, 3],

    # ── Event types ───────────────────────────────────────────────────────────
    "malicious_event_types": [
        "phishing_credential_stealer",
        "privilege_escalation",
        "exploit_cve",
        "failed_login_attempts",
        "suspicious_file_download",
        "lateral_movement",
        "execution_suspicious_process",
        "download_suspicious_domain",
        "c2_connection",
        "data_exfiltration",
    ],
    "benign_event_types": ["normal_activity"],

    # ── Feature generation parameters ─────────────────────────────────────────
    # 'means'       — reference center for each feature (a guide, not a fixed value).
    # 'mean_jitter' — std of a per-event mean shift applied per-event, simulating
    #                 context-dependent behaviour:
    #                   • a benign user repeatedly mistyping their password will have
    #                     temporarily elevated features that look suspicious
    #                   • a stealthy attacker mimicking normal activity will have
    #                     feature values that drift into the benign range
    # 'std'         — noise sampled around each event's (already-jittered) mean.
    #
    # Distributions are intentionally overlapping so the boundary is ambiguous,
    # forcing the model to learn a probabilistic decision boundary rather than
    # a trivial hard threshold.
    "feature_params": {
        "malicious": {"means": [0.55, 0.65, 0.70, 0.75, 0.85], "mean_jitter": 0.25, "std": 0.30},
        "benign":    {"means": [0.30, 0.40, 0.45, 0.50, 0.55], "mean_jitter": 0.22, "std": 0.28},
    },

    # ── Temporal settings ─────────────────────────────────────────────────────
    "start_timestamp":       "2025-04-13 12:00:00",
    "avg_interval_seconds":  60,   # average seconds between consecutive events

    # ── LLM backend ───────────────────────────────────────────────────────────
    # Choose: "synthetic" | "openai" | "cohere"
    # "synthetic" requires no API key and uses hardcoded realistic defaults.
    "llm_backend": "openai",

    "openai_model":   "gpt-4o-mini",
    "cohere_model":   "command-a-03-2025",

    # Set your API keys here or export them as environment variables:
    #   OPENAI_API_KEY  /  COHERE_API_KEY
    "openai_api_key": "",
    "cohere_api_key": "",

    # ── Misc ──────────────────────────────────────────────────────────────────
    "random_seed": 42,
    "output_path": "mads_synthetic_dataset.csv",


    # ── Checkpoint / resume ─────────────────────────────────────────────────────
    # Generation state is saved every 'checkpoint_interval' events.
    # Delete the checkpoint files to force a clean restart.
    "checkpoint_path":      "mads_checkpoint.json",
    "checkpoint_rows_path": "mads_checkpoint_rows.csv",
    "checkpoint_interval":  200,
}

In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# LLM BACKENDS
# ─────────────────────────────────────────────────────────────────────────────

class LLMBackend(ABC):
    """Abstract base class for all LLM backends."""

    @abstractmethod
    def generate(self, prompt: str) -> str:
        """Send a prompt and return the raw text response."""

    def generate_metadata_templates(self) -> dict:
        """
        Ask the LLM to produce a JSON object with realistic metadata.
        Returns a dict with keys: 'processes', 'users', 'ip_suffixes'.
        Falls back to built-in defaults on any parsing failure.
        """
        prompt = (
            "You are a cybersecurity data generator. "
            "Return ONLY a valid JSON object (no markdown, no extra text) with these keys:\n"
            "  'processes' : list of 25 realistic process names mixing legitimate "
            "(chrome.exe, svchost.exe, explorer.exe) and suspicious "
            "(mimikatz.exe, powershell.exe -enc, nc.exe, wmic.exe, certutil.exe).\n"
            "  'users'     : list of 20 user IDs formatted as U1001, U1002, …\n"
            "  'ip_suffixes': list of 30 distinct integers between 1 and 254 "
            "representing the last octet of 192.168.1.X addresses.\n"
            'Example: {"processes":["chrome.exe","mimikatz.exe"],'
            '"users":["U1001","U1002"],"ip_suffixes":[10,22,100]}'
        )
        try:
            raw = self.generate(prompt).strip()
            # Strip markdown fences if the model wrapped in ```json … ```
            if raw.startswith("```"):
                raw = raw.split("```")[1]
                if raw.startswith("json"):
                    raw = raw[4:]
            return json.loads(raw.strip())
        except Exception:
            return self._default_templates()

    @staticmethod
    def _default_templates() -> dict:
        return {
            "processes": [
                "powershell.exe", "cmd.exe", "chrome.exe", "svchost.exe",
                "proc_A", "proc_B", "proc_C", "python.exe", "wscript.exe",
                "mshta.exe", "explorer.exe", "rundll32.exe", "regsvr32.exe",
                "wmic.exe", "mimikatz.exe", "nc.exe", "certutil.exe",
                "bitsadmin.exe", "msiexec.exe", "cscript.exe",
            ],
            "users":       [f"U{1000 + i}" for i in range(1, 21)],
            "ip_suffixes": list(range(1, 31)),
        }


# ── Concrete backends ─────────────────────────────────────────────────────────

class OpenAIChatGPTBackend(LLMBackend):
    """OpenAI ChatGPT backend (requires `pip install openai`)."""

    def __init__(self, api_key: Optional[str] = None, model: str = "gpt-4o-mini"):
        import os
        import openai  # noqa: PLC0415
        self.client = openai.OpenAI(api_key=api_key or os.getenv("OPENAI_API_KEY"))
        self.model = model

    def generate(self, prompt: str) -> str:
        resp = self.client.chat.completions.create(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.7,
        )
        return resp.choices[0].message.content


class CohereCommandABackend(LLMBackend):
    """
    Cohere Command-A backend — command-a-03-2025 (requires `pip install cohere`).
    This is Cohere's flagship model from the Aya/Command family.
    """

    def __init__(self, api_key: Optional[str] = None, model: str = "command-a-03-2025"):
        import os
        import cohere  # noqa: PLC0415
        self.client = cohere.ClientV2(api_key=api_key or os.getenv("COHERE_API_KEY"))
        self.model = model

    def generate(self, prompt: str) -> str:
        resp = self.client.chat(
            model=self.model,
            messages=[{"role": "user", "content": prompt}],
        )
        return resp.message.content[0].text


class SyntheticBackend(LLMBackend):
    """
    No-LLM fallback.  Uses hard-coded realistic defaults — no API key needed.
    """

    def generate(self, prompt: str) -> str:  # noqa: ARG002
        return json.dumps(self._default_templates())

    def generate_metadata_templates(self) -> dict:
        return self._default_templates()


# ── Factory ───────────────────────────────────────────────────────────────────

def create_backend(config: dict) -> LLMBackend:
    """Instantiate the LLM backend specified in *config['llm_backend']*."""
    name = config.get("llm_backend", "synthetic").lower()
    if name == "openai":
        return OpenAIChatGPTBackend(
            api_key=config.get("openai_api_key"),
            model=config.get("openai_model", "gpt-4o-mini"),
        )
    if name == "cohere":
        return CohereCommandABackend(
            api_key=config.get("cohere_api_key"),
            model=config.get("cohere_model", "command-a-03-2025"),
        )
    if name != "synthetic":
        print(f"[WARN] Unknown backend '{name}'. Falling back to SyntheticBackend.")
    return SyntheticBackend()


print("LLM backend classes loaded.")

LLM backend classes loaded.


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# DATASET GENERATOR
# ─────────────────────────────────────────────────────────────────────────────

# Multistage attack chain order:
#   reconnaissance → delivery → execution → persistence → exfiltration
ATTACK_CHAIN = [
    "failed_login_attempts",        # reconnaissance
    "phishing_credential_stealer",  # delivery
    "exploit_cve",                  # execution
    "privilege_escalation",         # persistence
    "data_exfiltration",            # exfiltration
]


class MADSDatasetGenerator:
    """
    Generates synthetic MADS cybersecurity event datasets.

    Parameters
    ----------
    config   : configuration dict (see CONFIG cell)
    backend  : an LLMBackend instance used for metadata enrichment
    """

    def __init__(self, config: dict, backend: LLMBackend):
        self.config   = config
        self.backend  = backend
        self.rng      = np.random.default_rng(config.get("random_seed", 42))
        self._metadata: Optional[dict] = None   # cached after first fetch

    # ── Private helpers ───────────────────────────────────────────────────────

    def _get_metadata(self) -> dict:
        """Fetch (and cache) metadata templates from the backend."""
        if self._metadata is None:
            print("[INFO] Fetching metadata templates from LLM backend…")
            self._metadata = self.backend.generate_metadata_templates()
            print(
                f"[INFO] Metadata ready — "
                f"{len(self._metadata['processes'])} processes, "
                f"{len(self._metadata['users'])} users, "
                f"{len(self._metadata['ip_suffixes'])} IP suffixes."
            )
        return self._metadata

    def _dirichlet_machine_probs(self) -> np.ndarray:
        alphas = np.array(self.config["machine_dirichlet_alphas"], dtype=float)
        return self.rng.dirichlet(alphas)

    def _assign_machines(self, n: int) -> list:
        probs    = self._dirichlet_machine_probs()
        machines = self.config["machines"]
        return self.rng.choice(machines, size=n, p=probs).tolist()

    def _assign_label(self, machine: str) -> str:
        mal_prob = self.config["machine_malicious_probs"][machine]
        return "malicious" if self.rng.random() < mal_prob else "benign"

    def _assign_event_type(self, label: str) -> str:
        types = (
            self.config["malicious_event_types"]
            if label == "malicious"
            else self.config["benign_event_types"]
        )
        return str(self.rng.choice(types))

    def _generate_features(self, label: str) -> list:
        params = self.config["feature_params"][label]
        # Shift each reference mean by a per-event jitter so events of the
        # same class don't all cluster at identical feature centers.
        jitter      = params.get("mean_jitter", 0.0)
        event_means = [
            m + float(self.rng.normal(0, jitter))
            for m in params["means"]
        ]
        return [
            round(float(self.rng.normal(m, params["std"])), 4)
            for m in event_means
        ]

    def _generate_timestamps(self, n: int) -> list:
        start    = datetime.fromisoformat(self.config["start_timestamp"])
        scale    = self.config.get("avg_interval_seconds", 60)
        gaps     = self.rng.exponential(scale=scale, size=n)
        times, t = [], start
        for gap in gaps:
            times.append(t)
            t += timedelta(seconds=float(gap))
        return times

    def _inject_attack_sequences(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Replace consecutive malicious events on primary machines with
        the ATTACK_CHAIN sequence to simulate multistage attacks.
        """
        chain_len = len(ATTACK_CHAIN)
        for machine in ["machine1", "machine2", "machine3"]:
            mal_idx = df.index[
                (df["machine"] == machine) & (df["label"] == "malicious")
            ].tolist()
            if len(mal_idx) < chain_len:
                continue

            # machine1 gets two overlapping chain injections, others get one
            n_seqs = 2 if machine == "machine1" else 1
            used: set = set()
            for _ in range(n_seqs):
                avail = [
                    i for i in range(len(mal_idx) - chain_len + 1)
                    if not any(mal_idx[i + j] in used for j in range(chain_len))
                ]
                if not avail:
                    break
                start = random.choice(avail)
                for j, et in enumerate(ATTACK_CHAIN):
                    pos = mal_idx[start + j]
                    df.at[pos, "event_type"] = et
                    used.add(pos)
        return df

    # ── Checkpoint helpers ────────────────────────────────────────────────────

    def _save_checkpoint(self, rows: list, next_index: int,
                         timestamps: list, machines: list) -> None:
        """Persist current progress to disk."""
        state = {
            "next_index":  next_index,
            "n_total":     self.config["n_events"],
            "rng_state":   self.rng.bit_generator.state,
            "metadata":    self._metadata,
            "timestamps":  [ts if isinstance(ts, str) else ts.isoformat()
                            for ts in timestamps],
            "machines":    machines,
        }
        cp_json = self.config.get("checkpoint_path",      "mads_checkpoint.json")
        cp_csv  = self.config.get("checkpoint_rows_path", "mads_checkpoint_rows.csv")
        with open(cp_json, "w") as fh:
            json.dump(state, fh)
        pd.DataFrame(rows).to_csv(cp_csv, index=False)
        print(f"[CHECKPOINT] Saved progress at event {next_index}/{state['n_total']}")

    def _load_checkpoint(self) -> Optional[dict]:
        """Load a previous checkpoint if both files exist; None otherwise."""
        cp_json = self.config.get("checkpoint_path",      "mads_checkpoint.json")
        cp_csv  = self.config.get("checkpoint_rows_path", "mads_checkpoint_rows.csv")
        if not (os.path.exists(cp_json) and os.path.exists(cp_csv)):
            return None
        with open(cp_json) as fh:
            state = json.load(fh)
        state["rows"] = pd.read_csv(cp_csv).to_dict("records")
        return state

    def _clear_checkpoint(self) -> None:
        """Delete checkpoint files after a successful full generation."""
        for key in ("checkpoint_path", "checkpoint_rows_path"):
            path = self.config.get(key)
            if path and os.path.exists(path):
                os.remove(path)

    # ── Public API ────────────────────────────────────────────────────────────

    def generate(self, n: Optional[int] = None, resume: bool = True) -> pd.DataFrame:
        """
        Generate *n* synthetic security events.

        Parameters
        ----------
        n      : number of events (defaults to config['n_events'])
        resume : if True, automatically resume from a saved checkpoint
                 when one exists for the same *n*.

        Returns
        -------
        pd.DataFrame with columns:
            event_id, timestamp, machine, event_type,
            feature_1…5, ip_address, user_id, process, label
        """
        n        = n if n is not None else self.config["n_events"]
        interval = self.config.get("checkpoint_interval", 200)

        # ── Attempt resume ────────────────────────────────────────────────────
        checkpoint = self._load_checkpoint() if resume else None
        if checkpoint and checkpoint.get("n_total") != n:
            print(f"[WARN] Checkpoint was for n={checkpoint['n_total']}, "
                  f"current n={n}. Starting fresh.")
            checkpoint = None

        if checkpoint:
            done = checkpoint["next_index"]
            print(f"[INFO] Resuming from event {done}/{n} "
                  f"({done/n*100:.1f}% already done) …")
            self._metadata               = checkpoint["metadata"]
            self.rng.bit_generator.state = checkpoint["rng_state"]
            timestamps = [datetime.fromisoformat(ts)
                          for ts in checkpoint["timestamps"]]
            machines   = checkpoint["machines"]
            rows       = checkpoint["rows"]
            start_idx  = checkpoint["next_index"]
        else:
            print("[INFO] Starting fresh generation …")
            self._get_metadata()
            timestamps = self._generate_timestamps(n)
            machines   = self._assign_machines(n)
            rows       = []
            start_idx  = 0

        meta = self._get_metadata()   # always returns the cached value

        # ── Main loop with progress bar ───────────────────────────────────────
        with tqdm(total=n, initial=start_idx, desc="Generating events",
                  unit="evt", dynamic_ncols=True) as pbar:
            for i in range(start_idx, n):
                machine  = machines[i]
                ts       = timestamps[i]
                label    = self._assign_label(machine)
                features = self._generate_features(label)
                rows.append({
                    "event_id":   f"EVT-{i + 1:05d}",
                    "timestamp":  ts.isoformat(),
                    "machine":    machine,
                    "event_type": self._assign_event_type(label),
                    "feature_1":  features[0],
                    "feature_2":  features[1],
                    "feature_3":  features[2],
                    "feature_4":  features[3],
                    "feature_5":  features[4],
                    "ip_address": f"192.168.1.{self.rng.choice(meta['ip_suffixes'])}",
                    "user_id":    str(self.rng.choice(meta["users"])),
                    "process":    str(self.rng.choice(meta["processes"])),
                    "label":      label,
                })
                pbar.update(1)

                # Checkpoint periodically (skip on the very last event)
                if (i + 1) % interval == 0 and (i + 1) < n:
                    self._save_checkpoint(rows, i + 1, timestamps, machines)

        # ── Post-processing ───────────────────────────────────────────────────
        df = pd.DataFrame(rows)
        df["timestamp"] = pd.to_datetime(df["timestamp"], format="ISO8601")
        df = self._inject_attack_sequences(df)
        df = df.sort_values("timestamp").reset_index(drop=True)
        self._clear_checkpoint()
        return df


print("MADSDatasetGenerator loaded.")

MADSDatasetGenerator loaded.


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# UTILITY FUNCTIONS
# ─────────────────────────────────────────────────────────────────────────────

def display_dataset_stats(df: pd.DataFrame) -> None:
    """Print a concise summary of the generated dataset."""
    n = len(df)
    print("=" * 65)
    print(f"  Total events : {n}")
    print("-" * 65)
    print("  Label distribution:")
    for label, cnt in df["label"].value_counts().items():
        bar = "█" * int(cnt / n * 40)
        print(f"    {label:<12} {cnt:>5}  ({cnt/n*100:5.1f}%)  {bar}")
    print("-" * 65)
    print("  Events per machine (% malicious):")
    for machine in sorted(df["machine"].unique()):
        sub = df[df["machine"] == machine]
        mal = (sub["label"] == "malicious").sum()
        print(f"    {machine:<10} {len(sub):>5} events   {mal/len(sub)*100:5.1f}% malicious")
    print("-" * 65)
    print("  Top event types:")
    for et, cnt in df["event_type"].value_counts().head(6).items():
        print(f"    {et:<40} {cnt:>4}")
    print("=" * 65)


print("Utility functions loaded.")

Utility functions loaded.


## Preview: Inspect 10 Sample Events

Run the cell below to generate a **small batch of `PREVIEW_N` events** and visually inspect the schema, feature distributions, and metadata before committing to the full generation.

> If the sample looks correct, proceed to the **Full Dataset Generation** section below.

In [13]:
PREVIEW_N = 10

print(f"Generating {PREVIEW_N} sample events for manual inspection…")
print(f"Backend : {CONFIG['llm_backend'].upper()}\n")

# Instantiate backend and generator (the generator is reused later for the
# full run so that metadata templates are fetched only once).
_backend   = create_backend(CONFIG)
_generator = MADSDatasetGenerator(CONFIG, _backend)

preview_df = _generator.generate(n=PREVIEW_N, resume=False)

print(f"\n{'─'*45}")
print(f"{'Column':<14}  {'Non-null':>8}  {'Dtype':<12}")
print(f"{'─'*45}")
for col in preview_df.columns:
    print(f"  {col:<14} {preview_df[col].notna().sum():>6}  {str(preview_df[col].dtype):<12}")
print(f"{'─'*45}\n")

preview_df

Generating 10 sample events for manual inspection…
Backend : OPENAI

[INFO] Starting fresh generation …
[INFO] Fetching metadata templates from LLM backend…
[INFO] Metadata ready — 25 processes, 20 users, 30 IP suffixes.


Generating events:   0%|          | 0/10 [00:00<?, ?evt/s]


─────────────────────────────────────────────
Column          Non-null  Dtype       
─────────────────────────────────────────────
  event_id           10  object      
  timestamp          10  datetime64[ns]
  machine            10  object      
  event_type         10  object      
  feature_1          10  float64     
  feature_2          10  float64     
  feature_3          10  float64     
  feature_4          10  float64     
  feature_5          10  float64     
  ip_address         10  object      
  user_id            10  object      
  process            10  object      
  label              10  object      
─────────────────────────────────────────────



,event_id,timestamp,machine,event_type,feature_1,feature_2,feature_3,feature_4,feature_5,ip_address,user_id,process,label
0,EVT-00001,2025-04-13 12:00:00.000000,machine4,normal_activity,0.1787,0.0521,0.0401,0.8177,1.0065,192.168.1.5,U1017,java.exe,benign
1,EVT-00002,2025-04-13 12:02:24.252516,machine2,download_suspicious_domain,0.4507,0.9117,0.7494,0.8914,1.2572,192.168.1.15,U1015,wmic.exe,malicious
2,EVT-00003,2025-04-13 12:04:44.423895,machine5,normal_activity,-0.3118,0.2457,0.8245,0.4737,0.9622,192.168.1.24,U1016,certutil.exe,benign
3,EVT-00004,2025-04-13 12:07:07.509555,machine5,lateral_movement,0.1586,1.0136,0.6949,0.6382,0.4385,192.168.1.24,U1020,calc.exe,malicious
4,EVT-00005,2025-04-13 12:07:24.297212,machine4,suspicious_file_download,0.8103,0.3437,0.6130,0.7988,0.6850,192.168.1.14,U1004,cmd.exe,malicious
5,EVT-00006,2025-04-13 12:07:29.483456,machine1,exploit_cve,0.6500,-0.0435,0.2248,0.8091,0.4938,192.168.1.22,U1016,powershell.exe,malicious
6,EVT-00007,2025-04-13 12:08:56.643087,machine2,execution_suspicious_process,0.3838,0.3870,0.4763,0.3398,0.7998,192.168.1.13,U1007,java.exe,malicious
7,EVT-00008,2025-04-13 12:10:21.240729,machine1,exploit_cve,0.7407,0.5012,1.1838,0.4484,0.9979,192.168.1.28,U1020,python.exe,malicious
8,EVT-00009,2025-04-13 12:13:28.698486,machine1,lateral_movement,0.9602,1.1935,1.2065,0.6204,0.6315,192.168.1.15,U1017,wmic.exe,malicious
9,EVT-00010,2025-04-13 12:13:33.456138,machine3,download_suspicious_domain,0.3986,0.1863,0.1624,0.8706,0.8731,192.168.1.22,U1019,ssh.exe,malicious


## Full Dataset Generation

Satisfied with the preview? Run the cell below to generate the complete dataset (`n_events` from CONFIG) and display summary statistics.

In [14]:
print(f"Generating full dataset of {CONFIG['n_events']} events…")
print(f"Backend : {CONFIG['llm_backend'].upper()}\n")

# _generator already has metadata cached from the preview run.
# If you skipped the preview, _generator may not exist — create it here.
try:
    _generator
except NameError:
    _backend   = create_backend(CONFIG)
    _generator = MADSDatasetGenerator(CONFIG, _backend)

full_df = _generator.generate()

print()
display_dataset_stats(full_df)
full_df.head()

Generating full dataset of 6000 events…
Backend : OPENAI

[INFO] Starting fresh generation …


Generating events:   0%|          | 0/6000 [00:00<?, ?evt/s]

[CHECKPOINT] Saved progress at event 200/6000
[CHECKPOINT] Saved progress at event 400/6000
[CHECKPOINT] Saved progress at event 600/6000
[CHECKPOINT] Saved progress at event 800/6000
[CHECKPOINT] Saved progress at event 1000/6000
[CHECKPOINT] Saved progress at event 1200/6000
[CHECKPOINT] Saved progress at event 1400/6000
[CHECKPOINT] Saved progress at event 1600/6000
[CHECKPOINT] Saved progress at event 1800/6000
[CHECKPOINT] Saved progress at event 2000/6000
[CHECKPOINT] Saved progress at event 2200/6000
[CHECKPOINT] Saved progress at event 2400/6000
[CHECKPOINT] Saved progress at event 2600/6000
[CHECKPOINT] Saved progress at event 2800/6000
[CHECKPOINT] Saved progress at event 3000/6000
[CHECKPOINT] Saved progress at event 3200/6000
[CHECKPOINT] Saved progress at event 3400/6000
[CHECKPOINT] Saved progress at event 3600/6000
[CHECKPOINT] Saved progress at event 3800/6000
[CHECKPOINT] Saved progress at event 4000/6000
[CHECKPOINT] Saved progress at event 4200/6000
[CHECKPOINT] Save

,event_id,timestamp,machine,event_type,feature_1,feature_2,feature_3,feature_4,feature_5,ip_address,user_id,process,label
0,EVT-00001,2025-04-13 12:00:00.000000,machine1,data_exfiltration,0.7266,0.7384,-0.1083,0.7594,0.9460,192.168.1.20,U1013,python.exe,malicious
1,EVT-00002,2025-04-13 12:00:11.459098,machine3,normal_activity,0.4288,0.5284,0.4184,0.4572,0.8116,192.168.1.13,U1006,whoami.exe,benign
2,EVT-00003,2025-04-13 12:01:32.010817,machine4,suspicious_file_download,0.5295,0.7131,0.5481,0.9290,-0.0716,192.168.1.30,U1015,svchost.exe,malicious
3,EVT-00004,2025-04-13 12:02:30.953552,machine1,lateral_movement,0.4490,-0.0086,0.9424,0.4923,0.5986,192.168.1.15,U1013,rundll32.exe,malicious
4,EVT-00005,2025-04-13 12:02:48.231682,machine1,c2_connection,0.5861,0.5747,0.3330,0.4468,0.9615,192.168.1.23,U1008,netstat.exe,malicious


In [15]:
# ── Save to CSV ───────────────────────────────────────────────────────────────
output_path = CONFIG["output_path"]
full_df.to_csv(output_path, index=False)

print(f"Dataset saved  → {output_path}")
print(f"Shape          : {full_df.shape[0]} rows × {full_df.shape[1]} columns")

# ── Feature statistics by label ───────────────────────────────────────────────
print("\nFeature means by label:")
full_df.groupby("label")[["feature_1","feature_2","feature_3","feature_4","feature_5"]].mean().round(3)

Dataset saved  → mads_synthetic_dataset.csv
Shape          : 6000 rows × 13 columns

Feature means by label:


,feature_1,feature_2,feature_3,feature_4,feature_5
label,,,,,
benign,0.298,0.398,0.444,0.489,0.542
malicious,0.551,0.647,0.707,0.755,0.852
